# Dynamic Prompts with Runtime Context

Demonstrates how to use middleware and runtime context to customize agent behavior without external dependencies like a database.

Key concepts:
- `RuntimeContext` holds user-specific state (role, permissions, etc.)
- `@dynamic_prompt` middleware customizes the system prompt based on context
- Tools can access the runtime context to enforce permissions
- No database required—works with in-memory data

## Setup

Load environment variables

In [1]:
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

True

## Define Runtime Context

Context holds user-specific data accessible to tools and middleware.

In [2]:
from dataclasses import dataclass


@dataclass
class RuntimeContext:
    """User context that flows through the agent."""

    user_role: str  # "viewer", "editor", or "admin"
    documents: list[dict]  # In-memory document store


# Sample documents
SAMPLE_DOCUMENTS = [
    {
        "id": 1,
        "title": "Product Roadmap Q2 2026",
        "visibility": "public",
        "content": "Launch new API features in Q2...",
    },
    {
        "id": 2,
        "title": "Payroll Summary",
        "visibility": "internal",
        "content": "Employee compensation details...",
    },
    {
        "id": 3,
        "title": "Security Incident Report",
        "visibility": "admin_only",
        "content": "Breach detected in...",
    },
]

## Define Tools with Context Access

Tools can check the runtime context to enforce permissions.

In [3]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime


@tool
def search_documents(query: str) -> str:
    """Search documents by title or content.

    Returns only documents the user has permission to view based on their role.
    """
    runtime = get_runtime(RuntimeContext)
    user_role = runtime.context.user_role
    documents = runtime.context.documents

    # Filter documents by role permissions
    if user_role == "viewer":
        allowed_visibility = {"public"}
    elif user_role == "editor":
        allowed_visibility = {"public", "internal"}
    elif user_role == "admin":
        allowed_visibility = {"public", "internal", "admin_only"}
    else:
        return "Error: Unknown user role"

    # Search and filter
    results = []
    query_lower = query.lower()
    for doc in documents:
        if doc["visibility"] in allowed_visibility:
            if query_lower in doc["title"].lower() or query_lower in doc["content"].lower():
                results.append(
                    f"[{doc['id']}] {doc['title']} ({doc['visibility']})\n{doc['content'][:100]}..."
                )

    if not results:
        return "No documents found matching your query and access level."
    return "\n\n".join(results)

## Create Dynamic Prompt Template

The base template with a placeholder for role-specific instructions.

In [4]:
SYSTEM_PROMPT_TEMPLATE = """You are a helpful document assistant.

Rules:
- Help users find and understand documents.
- Only search for documents matching their role and permissions.
- The search_documents tool automatically filters by access level.
{role_instructions}
- If a search returns 'No documents found', it means they don't have access or the document doesn't exist.
"""

## Build Dynamic Prompt Middleware

Customize the prompt based on the runtime context.

In [5]:
from langchain.agents.middleware.types import ModelRequest, dynamic_prompt


@dynamic_prompt
def dynamic_system_prompt(request: ModelRequest) -> str:
    """Generate a prompt customized for the user's role."""
    user_role = request.runtime.context.user_role

    if user_role == "viewer":
        role_instructions = (
            "- You have READ-ONLY access to public documents.\n"
            "- Do not attempt to modify or access confidential information."
        )
    elif user_role == "editor":
        role_instructions = (
            "- You can access public and internal documents.\n"
            "- Help users understand internal workflows and processes."
        )
    elif user_role == "admin":
        role_instructions = (
            "- You have full access to all documents including admin-only content.\n"
            "- Help with sensitive security and operational matters."
        )
    else:
        role_instructions = "- Unknown role. Limited to public documents."

    return SYSTEM_PROMPT_TEMPLATE.format(role_instructions=role_instructions)

## Create Agent with Middleware

Include the dynamic prompt middleware in `create_agent` and specify the context schema.

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    model="claude-haiku-4-5",
    tools=[search_documents],
    middleware=[dynamic_system_prompt],
    context_schema=RuntimeContext,
)

## Test: Viewer Role (Limited Access)

A viewer can only see public documents.

In [7]:
question = "Can you find documents about security?"

print("\n=== VIEWER ROLE ===")
for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(user_role="viewer", documents=SAMPLE_DOCUMENTS),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()


=== VIEWER ROLE ===
================================ Human Message =================================

Can you find documents about security?
================================== Ai Message ==================================

[{'text': "I'll search for documents about security for you.", 'type': 'text'}, {'id': 'toolu_01FmFu1a9gBfevHcgjaroK3Y', 'caller': {'type': 'direct'}, 'input': {'query': 'security'}, 'name': 'search_documents', 'type': 'tool_use'}]
Tool Calls:
  search_documents (toolu_01FmFu1a9gBfevHcgjaroK3Y)
 Call ID: toolu_01FmFu1a9gBfevHcgjaroK3Y
  Args:
    query: security
================================= Tool Message =================================
Name: search_documents

No documents found matching your query and access level.
================================== Ai Message ==================================

It appears there are no documents about security that match your current access level, or such documents don't exist in the system. 

This could mean:
- Security-relat

## Test: Editor Role (Medium Access)

An editor can see public and internal documents.

In [8]:
print("\n=== EDITOR ROLE ===")
for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(user_role="editor", documents=SAMPLE_DOCUMENTS),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()


=== EDITOR ROLE ===
================================ Human Message =================================

Can you find documents about security?
================================== Ai Message ==================================

[{'text': "I'll search for documents about security for you.", 'type': 'text'}, {'id': 'toolu_0156fNpPgPfe6vQhbpT4erVo', 'caller': {'type': 'direct'}, 'input': {'query': 'security'}, 'name': 'search_documents', 'type': 'tool_use'}]
Tool Calls:
  search_documents (toolu_0156fNpPgPfe6vQhbpT4erVo)
 Call ID: toolu_0156fNpPgPfe6vQhbpT4erVo
  Args:
    query: security
================================= Tool Message =================================
Name: search_documents

No documents found matching your query and access level.
================================== Ai Message ==================================

It looks like there are no documents about security that match your current access level, or there may not be any security-related documents in the system.

Would you 

## Test: Admin Role (Full Access)

An admin can see all documents, including admin-only content.

In [9]:
print("\n=== ADMIN ROLE ===")
for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(user_role="admin", documents=SAMPLE_DOCUMENTS),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()


=== ADMIN ROLE ===
================================ Human Message =================================

Can you find documents about security?
================================== Ai Message ==================================

[{'text': "I'll search for documents about security for you.", 'type': 'text'}, {'id': 'toolu_01Hscnd6fBbj35UMXysTRtoN', 'caller': {'type': 'direct'}, 'input': {'query': 'security'}, 'name': 'search_documents', 'type': 'tool_use'}]
Tool Calls:
  search_documents (toolu_01Hscnd6fBbj35UMXysTRtoN)
 Call ID: toolu_01Hscnd6fBbj35UMXysTRtoN
  Args:
    query: security
================================= Tool Message =================================
Name: search_documents

[3] Security Incident Report (admin_only)
Breach detected in......
================================== Ai Message ==================================

I found a document related to security:

**Security Incident Report** (admin-only access)
- This appears to be an incident report involving a breach detection

## Key Takeaways

1. **RuntimeContext** holds user-specific state (role, permissions, data)
2. **@dynamic_prompt middleware** customizes the system prompt at request time
3. **Tools access context** via `get_runtime(RuntimeContext)` to enforce permissions
4. **No database required** — works with in-memory data, APIs, or any data source
5. **Role-based behavior** — viewer sees nothing sensitive, admin sees everything

This pattern scales to:
- User preference-based prompts (language, detail level, tone)
- Feature flags (enable/disable tool access by context)
- Rate limiting (tool behaviors based on quota context)
- Multi-tenant isolation (context per tenant)